# Implementing Guardrail in an Agent


In [ ]:
from agents import Agent, Runner, output_guardrail, GuardrailFunctionOutput  # noqa: I001
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown
from rich.syntax import Syntax  # noqa: F401

console = Console()
OPEANAI_MODEL_NAME = "gpt-5.4-nano"


In [19]:
class Email_review(BaseModel):
    is_professional: bool = Field(description="Checks if the email tone is professional")
    num_lines: int = Field(description="Captures the number of lines in the email")
    has_placeholder: bool = Field(description="Checks if the email contains any placeheolder")

In [20]:
instructions = """
Review the potential sales emails
"""

email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

AG
"""

In [21]:
email_reviewer = Agent(
    name="Email Reviewer",
    instructions=instructions,
    output_type=Email_review,
    model=OPEANAI_MODEL_NAME,
)

In [22]:
result = await Runner.run(email_reviewer, input=email)

### Create Guardrail function

Review the final output passed to the callback as `message`. Trigger the guardrail if the email is unprofessional **or** contains placeholders. A triggered output guardrail raises `OutputGuardrailTripwireTriggered` after generation; it does not prevent generation itself.


In [26]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(email_reviewer, input=message, context=ctx.context)
    review = result.final_output
    is_problem = not review.is_professional or review.has_placeholder
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [27]:
instructions = instructions + "\nSpeak like a cowboy"
agent2 = Agent(
    name="Cowboy Agent",
    instructions=instructions,
    model=OPEANAI_MODEL_NAME,
    output_guardrails=[email_guardrail],
)

In [28]:
result = await Runner.run(agent2, input="Write cold sales email")

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

In [ ]:
console.print(Markdown(result.final_output))